# Pilote EdS — à exécuter dans Google Colab

Pilote sur **3 dossiers** du canevas : le premier sert au calibrage du seuil d'appariement, les deux
autres à la mesure. 3 dossiers × 3 modèles × 2 conditions × 5 répétitions = **90 appels**.

**Avant d'exécuter :**
- autorisation écrite d'exploitation des dossiers obtenue ;
- canevas **anonymisé** (identifiants EDS-xx, aucun nom de centre ni de participant) ;
- clés `ANTHROPIC_API_KEY`, `OPENAI_API_KEY`, `GOOGLE_API_KEY` dans le gestionnaire de secrets de Colab (icône clé) ;
- **ne jamais partager ce carnet une fois exécuté** : ses sorties contiennent le contenu des dossiers.

## 1. Installation, Drive, clés

In [ ]:
%pip install -q "hazid[providers] @ git+https://github.com/MoustaphaRouf/llm-hazard-id-eval"
from google.colab import drive, userdata
import os, pathlib
drive.mount('/content/drive')

BASE_DIR = pathlib.Path('/content/drive/MyDrive/ACADEMIE/AI')   # dossier contenant le canevas
for k in ['ANTHROPIC_API_KEY', 'OPENAI_API_KEY', 'GOOGLE_API_KEY']:
    try:
        os.environ[k] = userdata.get(k); print('OK      ', k)
    except Exception:
        print('ABSENTE ', k)

## 2. Canevas et choix des dossiers du pilote

In [ ]:
import pandas as pd
from hazid import adjudication, canvas, contamination, generation, matching
from hazid.protocol import Protocol

dossiers = canvas.load_canvas(BASE_DIR / 'canevas-synthese-eds.xlsx')
etat = pd.DataFrame([{'Dossier': k, 'Contexte (car.)': len(d.context), 'Dangers panel': len(d.human_hazards)}
                     for k, d in sorted(dossiers.items())])
display(etat)

complets = [k for k, d in sorted(dossiers.items()) if len(d.human_hazards) >= 3 and len(d.context) >= 300]
assert len(complets) >= 3, (f"Seulement {len(complets)} dossier(s) complet(s) : renseigner au moins 3 dossiers "
                            "(contexte pré-séance + au moins 3 dangers du panel) dans le canevas.")
CALIBRAGE = complets[0]        # sert uniquement à fixer le seuil
MESURE = complets[1:3]         # dossiers sur lesquels porteront les résultats
pilote = {k: dossiers[k] for k in [CALIBRAGE, *MESURE]}
print('Calibrage :', CALIBRAGE, '| Mesure :', MESURE)

## 3. Contrôle anti-contamination
Toute alerte doit être levée avant de poursuivre.

In [ ]:
alertes = contamination.check(pilote)
display(pd.DataFrame(alertes)) if alertes else print('Aucune contamination manifeste détectée.')
assert not alertes, 'Lever les alertes avant de poursuivre.'

## 4. Gel du protocole

Les identifiants doivent être des **versions exactes** (pas d'alias `latest`). Corriger `MODELES`
jusqu'à ce que la vérification soit entièrement OK, puis consigner l'empreinte au journal de l'étude.

In [ ]:
MODELES = {
    'A': {'provider': 'anthropic', 'model': 'claude-sonnet-4-6'},
    'B': {'provider': 'openai',    'model': 'gpt-5.1'},
    'C': {'provider': 'google',    'model': 'gemini-2.5-pro'},
}
protocole = Protocol(models=MODELES, temperature=0.7, n_runs=5)
ok = generation.check_models(protocole, list(MODELES))
assert ok, 'Corriger les identifiants de modèles.'
print("Empreinte du protocole (au journal) :", protocole.fingerprint())

## 5. Test à blanc — un dossier, un modèle, une exécution

In [ ]:
essai = Protocol(models=MODELES, temperature=0.7, n_runs=1)
generation.run_campaign({CALIBRAGE: pilote[CALIBRAGE]}, essai, BASE_DIR / 'pilote_test_blanc',
                        conditions=('blind',), keys=['A'])
r = generation.load_runs(BASE_DIR / 'pilote_test_blanc', ok_only=False)[0]
print(r['status'], r['n_hazards'], 'dangers', r['error'] or '')
assert r['status'] == 'ok', 'Corriger avant la campagne.'

## 6. Campagne pilote (90 appels, reprenable)

In [ ]:
RUNS = BASE_DIR / 'pilote_executions'
compte = generation.run_campaign(pilote, protocole, RUNS)
print(compte)
runs = generation.load_runs(RUNS)
print(len(runs), 'exécutions valides')

## 7. Calibrage du seuil — sur le dossier de calibrage uniquement

Examiner à la main les appariements proposés pour chaque seuil, choisir, puis **figer** `SEUIL`.
Ne pas regarder les résultats des dossiers de mesure avant.

In [ ]:
humains = {k: d.human_labels for k, d in pilote.items()}
display(matching.calibration_table(runs, humains, [CALIBRAGE]))

# Appariements proposés au seuil examiné, pour vérification manuelle
SEUIL_EXAMINE = 0.55
libs_m = sorted({x for r in runs if r['dossier'] == CALIBRAGE for x in matching.labels(r)})
for i, (j, s) in matching.match(humains[CALIBRAGE], libs_m, SEUIL_EXAMINE).items():
    print(f"{s:.2f} | PANEL : {humains[CALIBRAGE][i][:80]}\n     | MODÈLE: {libs_m[j][:80]}\n")

In [ ]:
SEUIL = None   # ← OBLIGATOIRE : valeur retenue après examen ci-dessus
assert SEUIL is not None, 'Figer le seuil avant de continuer.'
protocole.match_threshold = SEUIL
print('Empreinte finale (au journal) :', protocole.fingerprint())

## 8. Résultats sur les dossiers de mesure

In [ ]:
evaluation = [r for r in runs if r['dossier'] in MESURE]
indicateurs = matching.metrics(evaluation, humains, SEUIL)
accord = matching.agreement(evaluation, SEUIL)
display(indicateurs); display(accord)
indicateurs.to_csv(BASE_DIR / 'pilote_indicateurs.csv', index=False)
accord.to_csv(BASE_DIR / 'pilote_accord.csv', index=False)

## 9. Classeur d'adjudication en aveugle

Transmettre aux adjudicateurs (des pairs qui ne connaissent pas les dossiers) **uniquement**
`pilote-adjudication-a-transmettre.xlsx`. Garder la clé jusqu'à restitution complète des jugements.

In [ ]:
complet = BASE_DIR / 'pilote-adjudication-aveugle.xlsx'
transmis = BASE_DIR / 'pilote-adjudication-a-transmettre.xlsx'
n = adjudication.write_workbook(evaluation, {k: pilote[k].human_hazards for k in MESURE},
                                complet, protocole.shuffle_seed, collapse_threshold=SEUIL)
adjudication.export_for_adjudicators(complet, transmis)
print(n, 'lignes à adjudiquer ->', transmis.name)